# Bab 3 — Inertia and Within-Cluster Sum of Squares (WCSS)

Notebook pendamping Bab 3 Level 5.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Menjelaskan apa itu *Inertia* dan *Within-Cluster Sum of Squares* (WCSS), dan hubungannya dengan K-Means.
- Menghitung WCSS secara manual, titik demi titik, dan mencocokkannya dengan `inertia_` scikit-learn.
- Menjelaskan mengapa WCSS memakai jarak **kuadrat** ke centroid.
- Menjelaskan mengapa Inertia turun (atau tetap) saat K naik, dan mengapa Inertia terendah bukan berarti hasil terbaik.
- Menjelaskan pengaruh skala dan jumlah data pada nilai Inertia mentah.
- Menjelaskan mengapa Inertia bukan metrik universal untuk semua algoritma clustering.
- Menggambar grafik Inertia terhadap K.
- Mengenali kesalahan umum saat memakai Inertia.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler

## Bagian B: Enam titik dan dua kelompoknya

In [ ]:
X_tiny = np.array(
    [
        [1, 1], [2, 1], [1, 2],   # kelompok 0
        [6, 5], [7, 5], [6, 6],   # kelompok 1
    ],
    dtype=float,
)
labels_tiny = np.array([0, 0, 0, 1, 1, 1])

## Bagian C: WCSS dihitung titik demi titik

In [ ]:
total = 0.0
for k in (0, 1):
    members = X_tiny[labels_tiny == k]
    centroid = members.mean(axis=0)
    print(f"Kelompok {k}, centroid = {centroid.round(3).tolist()}")
    for point in members:
        sq_dist = ((point - centroid) ** 2).sum()   # jarak kuadrat titik ke centroid
        total += sq_dist
        print(f"  titik {point.tolist()} -> jarak kuadrat = {sq_dist:.3f}")
print(f"WCSS manual = {total:.3f}")

## Bagian D: Mencocokkan dengan inertia_ scikit-learn

In [ ]:
km_tiny = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_tiny)
print("Label K-Means      :", km_tiny.labels_.tolist())
print(f"inertia_ scikit-learn = {km_tiny.inertia_:.3f}")

## Bagian E: WCSS sebagai fungsi

In [ ]:
def wcss(X, labels):
    """WCSS: jumlah kuadrat jarak setiap titik ke centroid kelompoknya."""
    total = 0.0
    for k in np.unique(labels):
        members = X[labels == k]
        centroid = members.mean(axis=0)
        total += ((members - centroid) ** 2).sum()
    return total


print(f"wcss(X_tiny, labels_tiny) = {wcss(X_tiny, labels_tiny):.3f}")

## Bagian F: Inertia untuk K = 1 sampai 10

In [ ]:
X_blobs, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.0, random_state=42)

k_values = list(range(1, 11))
inertias = []
for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_blobs)
    inertias.append(model.inertia_)

table = pd.DataFrame({"K": k_values, "Inertia": inertias})
table["Penurunan"] = -table["Inertia"].diff()
print(table.round(2).to_string(index=False))
print("Inertia tidak pernah naik saat K naik?", bool(np.all(np.diff(inertias) <= 1e-9)))

## Bagian G: Ketika K sama dengan jumlah titik

In [ ]:
# K sama dengan jumlah titik: tiap titik menjadi kelompoknya sendiri
km_all = KMeans(n_clusters=len(X_tiny), n_init=10, random_state=42).fit(X_tiny)
print(f"K = {len(X_tiny)} pada {len(X_tiny)} titik -> Inertia = {km_all.inertia_:.3f}")

## Bagian H: Menggambar Inertia terhadap K

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(k_values, inertias, marker="o", color=palette[0])
ax.set_xlabel("Jumlah kelompok (K)")
ax.set_ylabel("Inertia (WCSS)")
ax.set_title("Inertia selalu turun saat K naik")
ax.set_xticks(k_values)
plt.tight_layout()
plt.show()

## Bagian I: Inertia dipengaruhi skala dan jumlah data

In [ ]:
inertia_base = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_blobs).inertia_

# 1) Skala data dikali 10: semua jarak ikut dikali 10, jarak kuadrat dikali 100
inertia_x10 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_blobs * 10).inertia_

# 2) Jumlah data digandakan (data yang sama dua kali): Inertia ikut naik
X_double = np.vstack([X_blobs, X_blobs])
inertia_double = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_double).inertia_

print(f"Inertia data asli          : {inertia_base:.2f}")
print(f"Inertia data x 10          : {inertia_x10:.2f}  (rasio {inertia_x10 / inertia_base:.1f})")
print(f"Inertia data digandakan    : {inertia_double:.2f}  (rasio {inertia_double / inertia_base:.2f})")
print(f"Inertia per titik (asli)   : {inertia_base / len(X_blobs):.3f}")
print(f"Inertia per titik (ganda)  : {inertia_double / len(X_double):.3f}")

## Bagian J: Inertia hanya untuk objektif centroid

In [ ]:
dbscan = DBSCAN(eps=0.7, min_samples=5).fit(X_blobs)
print("KMeans punya inertia_ :", hasattr(KMeans(n_clusters=2).fit(X_blobs), "inertia_"))
print("DBSCAN punya inertia_ :", hasattr(dbscan, "inertia_"))

## Bagian K: Inertia lebih rendah belum tentu lebih baik

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.06, random_state=42)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moons)

print(f"Inertia K-Means (label salah bentuk) : {km_moons.inertia_:.2f}")
print(f"WCSS label benar (dua bulan utuh)    : {wcss(X_moons, y_moons):.2f}")

## Bagian L: Menggambar moons

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
panels = [
    (km_moons.labels_, f"Hasil K-Means\nInertia = {km_moons.inertia_:.1f}"),
    (y_moons, f"Label benar\nWCSS = {wcss(X_moons, y_moons):.1f}"),
]
for ax, (labels, title) in zip(axes, panels):
    for k in range(2):
        ax.scatter(X_moons[labels == k, 0], X_moons[labels == k, 1], s=14, color=palette[k])
    ax.set_title(title)
    ax.set_xlabel("Fitur 1")
axes[0].set_ylabel("Fitur 2")
plt.tight_layout()
plt.show()

## Bagian M: Inertia pada Mall Customers

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall = StandardScaler().fit_transform(df[features])

mall_inertia = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_mall).inertia_ for k in range(1, 11)]
mall_table = pd.DataFrame({"K": range(1, 11), "Inertia": mall_inertia})
mall_table["Penurunan"] = -mall_table["Inertia"].diff()
print(mall_table.round(2).to_string(index=False))

## Latihan

1. Hitung manual WCSS data satu dimensi \( 1, 2, 6, 7, 8 \) dengan label \( [0, 0, 1, 1, 1] \).
2. Jalankan K-Means pada data tersebut dengan K = 1, 2, dan 3. Berapa Inertia-nya?
3. Ubah `cluster_std=1.0` menjadi `2.0` pada `X_blobs`. Berapa Inertia dengan K = 4? Mengapa lebih besar?
4. Standarkan `X_blobs` dengan `StandardScaler`. Berapa Inertia untuk K = 1 dan K = 4?
5. Hitung Inertia per titik untuk Mall Customers pada K = 5.
6. Mengapa Inertia untuk K = 10 tidak bisa dipakai untuk menyimpulkan bahwa 10 kelompok lebih baik daripada 4?
7. Mengapa kita tidak boleh membandingkan Inertia data A (skala ribuan) dengan Inertia data B (skala satuan)?

Jawaban lengkap ada di halaman bab.

## Ringkasan

- **Inertia** (WCSS) adalah jumlah kuadrat jarak setiap titik ke centroid kelompoknya. Pada K-Means, inilah yang diminimalkan algoritma.
- Dihitung manual pada enam titik: **2,667**, sama dengan `inertia_` scikit-learn.
- Pada data yang sama dan tujuan K-Means standar, Inertia **turun atau tetap** saat K naik, dan mencapai 0 pada K = n. Karena itu Inertia terendah **bukan** cara memilih K.
- Nilai mentah dipengaruhi **skala** (dikali \( c^2 \)) dan **jumlah data**. Pada data kita, skala × 10 memberi rasio 100,0 dan data digandakan memberi rasio 2,00.
- Inertia **bukan metrik universal**: `DBSCAN` tidak punya `inertia_`, dan pada moons label benar memiliki WCSS (180,93) lebih tinggi daripada K-Means yang salah bentuk (121,63).
- Pada Mall Customers, penurunan Inertia melandai setelah K = 5 (43,35 lalu 10,51).